# Unsway — Phase 6E one-shot confirmatory test

Run this only after Phase 6D has frozen a candidate. It scores exactly that candidate and its matched random control on the disjoint replacement holdout, once.

In [ ]:
from pathlib import Path

repo = Path("/content/Unsway")
if (repo / ".git").is_dir():
    !git -C /content/Unsway pull --ff-only
else:
    !git clone https://github.com/idris404/Unsway.git /content/Unsway
%cd /content/Unsway
!pip install -q -e '.[dev]'

In [ ]:
import json

import torch

assert torch.cuda.is_available(), "Select a GPU runtime."
validation = json.loads(Path("reports/phase6d_validation.json").read_text())
assert validation["status"] == "confirmatory_candidate_frozen"
assert validation["test_prompts_scored"] is False
print("Frozen candidate:", validation["selected_confirmatory_candidate"])

The next two commands are the irreversible scientific decision point: the first binds all hashes before test access; the second opens pressure/control prompts exactly once.

In [ ]:
!python -m unsway.cli.phase6 \
    --config configs/phase6c.yaml --stage phase6e-freeze \
    --confirmatory-config configs/phase6e.yaml
!python -m unsway.cli.phase6 \
    --config configs/phase6c.yaml --stage phase6e-test \
    --confirmatory-config configs/phase6e.yaml

In [ ]:
result = json.loads(Path("reports/phase6e_test.json").read_text())
summary = {
    "status": result["status"],
    "baseline": result["baseline"],
    "candidate_delta": result["candidate"]["delta"],
    "candidate_bootstrap": result["candidate"]["bootstrap"],
    "source_deltas": result["candidate"]["source_pressure_effect_deltas"],
    "matched_random_delta": result["matched_random_control"]["delta"],
}
print(json.dumps(summary, indent=2))